# Chapter 3 — CNN Building Blocks: Pooling, Activations, BatchNorm, Dropout

## Learning Objectives
- Understand MaxPool, AvgPool, and Global Average Pooling
- Compare activation functions: ReLU, LeakyReLU, GELU, Sigmoid
- Understand Batch Normalisation: why it works, how to use it
- Understand Dropout as a regulariser
- Build a complete ConvBlock from scratch
- Train SimpleCNN on EuroSAT and compare architectures

## Estimated Duration: Theory 2h | Practical 2h | Total 4h
## Difficulty: Beginner

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q torch torchvision torchgeo matplotlib seaborn

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FAST_MODE = DEVICE == 'cpu'
DATA_ROOT = Path('./data') if not IN_COLAB else Path('/content/data')
print(f'Device: {DEVICE} | Fast mode: {FAST_MODE}')

## 3.1 — Pooling Layers

Pooling reduces spatial dimensions without learnable parameters:

MaxPool2d(2,2): takes maximum in each 2×2 window → spatial size ÷2
  - Preserves the strongest feature responses
  - Translation invariant: small shifts don't change the max
  - Standard in classification networks

AvgPool2d(2,2): takes average in each 2×2 window → spatial size ÷2
  - Smoother downsampling
  - Used in some segmentation decoders

GlobalAveragePool (GAP): reduces (B, C, H, W) → (B, C)
  - Eliminates fully-connected layers
  - Fewer parameters, stronger regularisation
  - Required for Class Activation Maps (CAM)
  - Standard in all modern architectures (ResNet, EfficientNet)

In [ ]:
# MaxPool vs AvgPool visual comparison
from torchgeo.datasets import EuroSAT

EUROSAT_ROOT = DATA_ROOT / 'eurosat'
EUROSAT_ROOT.mkdir(parents=True, exist_ok=True)
dataset = EuroSAT(root=EUROSAT_ROOT, split='train', download=True)

sample = dataset[42]['image'].float() / 10000.0
img = sample[:3].unsqueeze(0)  # (1, 3, 64, 64)

max_pool_2 = nn.MaxPool2d(2, 2)
avg_pool_2 = nn.AvgPool2d(2, 2)
max_pool_4 = nn.MaxPool2d(4, 4)

with torch.no_grad():
    mp2 = max_pool_2(img)
    ap2 = avg_pool_2(img)
    mp4 = max_pool_4(img)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
titles = ['Original (64×64)', 'MaxPool(2,2) → 32×32', 'AvgPool(2,2) → 32×32', 'MaxPool(4,4) → 16×16']
tensors = [img, mp2, ap2, mp4]

for ax, t, title in zip(axes, tensors, titles):
    disp = t.squeeze().permute(1, 2, 0).numpy().clip(0, 1)
    ax.imshow(disp)
    ax.set_title(title, fontsize=9)
    ax.axis('off')

plt.suptitle('Pooling Operations: Spatial Downsampling', fontsize=11)
plt.tight_layout()
plt.show()

print(f'MaxPool vs AvgPool on texture: '
      f'Max preserves edges; Avg is smoother')
print(f'Input: {img.shape} → MaxPool2: {mp2.shape} → MaxPool4: {mp4.shape}')

## 3.2 — Activation Functions

Without non-linear activations, a stack of conv layers collapses to a
single linear transformation — useless for complex patterns.

ReLU(x) = max(0, x)
  - Simple, fast, gradient = 1 for x > 0
  - 'Dying ReLU': neurons can get stuck at 0 (gradient = 0 for x ≤ 0)
  - Default choice for CNNs

LeakyReLU(x) = x if x > 0 else 0.01x
  - Fixes dying ReLU with small negative slope

GELU(x) = x * Φ(x)  (Gaussian CDF)
  - Smooth activation, preferred in transformers, also used in EfficientNet

Sigmoid: squashes to (0,1) — used ONLY in final binary classification layer
  - Saturates (gradient→0) for |x| >> 0 — do NOT use in hidden layers

Softmax: normalised exponentials over classes — output layer for multiclass

In [ ]:
# Visualise activation functions
x = torch.linspace(-4, 4, 200)

activations = {
    'ReLU': F.relu(x),
    'LeakyReLU(0.1)': F.leaky_relu(x, 0.1),
    'GELU': F.gelu(x),
    'Sigmoid': torch.sigmoid(x),
    'Tanh': torch.tanh(x),
    'SiLU (Swish)': x * torch.sigmoid(x),
}

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
colors = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0', '#795548', '#009688']

for ax, (name, y), color in zip(axes.ravel(), activations.items(), colors):
    ax.plot(x.numpy(), y.numpy(), color=color, linewidth=2)
    ax.axhline(0, color='k', linestyle='-', linewidth=0.5)
    ax.axvline(0, color='k', linestyle='-', linewidth=0.5)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlim(-4, 4)
    ax.grid(True, alpha=0.3)
    
    # Show gradient magnitude
    x_t = x.clone().requires_grad_(True)
    if name == 'ReLU': y_t = F.relu(x_t)
    elif 'Leaky' in name: y_t = F.leaky_relu(x_t, 0.1)
    elif name == 'GELU': y_t = F.gelu(x_t)
    elif name == 'Sigmoid': y_t = torch.sigmoid(x_t)
    elif name == 'Tanh': y_t = torch.tanh(x_t)
    else: y_t = x_t * torch.sigmoid(x_t)
    
    y_t.sum().backward()
    grad = x_t.grad.numpy()
    ax2 = ax.twinx()
    ax2.plot(x.numpy(), grad, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    ax2.set_ylabel('Gradient', color='gray', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='gray', labelsize=7)

plt.suptitle('Activation Functions and Their Gradients (solid=activation, dashed=gradient)', fontsize=12)
plt.tight_layout()
plt.show()

print('Note: ReLU gradient = 0 for x < 0 (dying ReLU problem)')
print('GELU has smooth gradient everywhere — preferred in modern architectures')

## 3.3 — Batch Normalisation

BatchNorm normalises the activations within each mini-batch:
  x̂ = (x - μ_batch) / (σ_batch + ε)   then scale:  y = γ·x̂ + β

Where:
  μ_batch = batch mean  (computed per channel)
  σ_batch = batch std   (computed per channel)
  γ, β    = learnable scale and shift (allows the network to undo normalisation)

Benefits:
  1. Reduces internal covariate shift → higher learning rates possible
  2. Acts as regulariser → less need for Dropout
  3. Gradient flow: normalisation prevents exploding/vanishing gradients

At inference: uses running mean/std accumulated during training.

EO note: BatchNorm is crucial with multispectral data because different
bands have very different value ranges (SWIR DN >> Blue DN).

In [ ]:
# Demonstrate BatchNorm stabilising training
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms

# Simple CNN without BN
class CNN_NoBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 10)
        )
    def forward(self, x): return self.net(x)

# Simple CNN with BN
class CNN_WithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 10)
        )
    def forward(self, x): return self.net(x)

# Count parameters
m_nobn = CNN_NoBN()
m_bn = CNN_WithBN()
print(f'CNN without BN: {sum(p.numel() for p in m_nobn.parameters()):,} parameters')
print(f'CNN with BN:    {sum(p.numel() for p in m_bn.parameters()):,} parameters')
print(f'BN overhead:    {sum(p.numel() for p in m_bn.parameters()) - sum(p.numel() for p in m_nobn.parameters()):,} params (γ,β per channel)')

In [ ]:
# Quick training comparison: with vs without BatchNorm
from torchvision import transforms as T

def make_loader(split='train', batch_size=64, fast=False):
    ds = EuroSAT(root=EUROSAT_ROOT, split=split, download=True)
    if fast:  # subset for CPU demo
        indices = list(range(0, len(ds), 5))  # 20% of data
        from torch.utils.data import Subset
        ds = Subset(ds, indices)
    return DataLoader(ds, batch_size=batch_size, shuffle=(split=='train'), num_workers=0)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        imgs = batch['image'].float() / 10000.0  # normalize to [0,1]
        labels = batch['label']
        imgs, labels = imgs[:, :3].to(device), labels.to(device)  # RGB only
        optimizer.zero_grad(set_to_none=True)
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(imgs)
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(imgs)
    return total_loss / total, correct / total

def eval_model(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            imgs = batch['image'].float() / 10000.0
            labels = batch['label']
            imgs, labels = imgs[:, :3].to(device), labels.to(device)
            logits = model(imgs)
            loss = criterion(logits, labels)
            total_loss += loss.item() * len(imgs)
            correct += (logits.argmax(1) == labels).sum().item()
            total += len(imgs)
    return total_loss / total, correct / total

train_loader = make_loader('train', batch_size=64, fast=FAST_MODE)
val_loader = make_loader('val', batch_size=64, fast=FAST_MODE)

n_epochs = 3 if FAST_MODE else 10
criterion = nn.CrossEntropyLoss()

models_to_compare = {
    'No BatchNorm': CNN_NoBN().to(DEVICE),
    'With BatchNorm': CNN_WithBN().to(DEVICE),
}

histories = {}
for name, model in models_to_compare.items():
    print(f'\nTraining: {name}')
    opt = optim.Adam(model.parameters(), lr=1e-3)
    hist = {'train_loss': [], 'val_acc': []}
    for epoch in range(1, n_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, opt, criterion, DEVICE)
        val_loss, val_acc = eval_model(model, val_loader, criterion, DEVICE)
        hist['train_loss'].append(tr_loss)
        hist['val_acc'].append(val_acc)
        print(f'  Epoch {epoch}/{n_epochs}: train_loss={tr_loss:.4f} val_acc={val_acc:.3f}')
    histories[name] = hist

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
colors = {'No BatchNorm': '#F44336', 'With BatchNorm': '#2196F3'}
for name, hist in histories.items():
    epochs = range(1, len(hist['train_loss']) + 1)
    ax1.plot(epochs, hist['train_loss'], label=name, color=colors[name])
    ax2.plot(epochs, hist['val_acc'], label=name, color=colors[name])

ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_title('Validation Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('Effect of BatchNorm on Training Stability', fontsize=12)
plt.tight_layout()
plt.show()

## 3.4 — Dropout: Regularisation by Random Deactivation

During training, Dropout randomly sets a fraction `p` of neuron outputs to 0.
At inference, all neurons are active but scaled by (1-p).

Why it works:
  - Prevents co-adaptation: neurons can't rely on specific partners
  - Approximates ensemble of 2^N different networks
  - Most effective in fully-connected layers

In modern CNNs:
  - Dropout(0.5) before the final FC layer is standard
  - SpatialDropout2d (drops entire feature map channels) works better in conv layers
  - BatchNorm provides implicit regularisation, reducing Dropout needs
  - DropPath/StochasticDepth is used in vision transformers

In [ ]:
# Demonstrate overfitting and how Dropout prevents it
# Use a tiny subset to force overfitting

from torch.utils.data import Subset

# Use only 500 training samples → force overfitting
tiny_train = Subset(EuroSAT(root=EUROSAT_ROOT, split='train', download=True), range(500))
tiny_loader = DataLoader(tiny_train, batch_size=32, shuffle=True, num_workers=0)

class CNN_NoDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(64, 10)  # No dropout
        )
    def forward(self, x): return self.net(x)

class CNN_WithDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(p),
            nn.Linear(64, 10)  # With dropout
        )
    def forward(self, x): return self.net(x)

n_epochs_overfit = 5 if FAST_MODE else 20
reg_models = {
    'No Dropout': CNN_NoDropout().to(DEVICE),
    'Dropout(0.5)': CNN_WithDropout(0.5).to(DEVICE),
}

reg_histories = {}
for name, model in reg_models.items():
    print(f'Training: {name}')
    opt = optim.Adam(model.parameters(), lr=1e-3)
    hist = {'train_acc': [], 'val_acc': []}
    for epoch in range(1, n_epochs_overfit + 1):
        _, tr_acc = train_one_epoch(model, tiny_loader, opt, criterion, DEVICE)
        _, val_acc = eval_model(model, val_loader, criterion, DEVICE)
        hist['train_acc'].append(tr_acc)
        hist['val_acc'].append(val_acc)
    reg_histories[name] = hist
    print(f'  Final: train_acc={hist["train_acc"][-1]:.3f}, val_acc={hist["val_acc"][-1]:.3f}')

# Plot overfitting gap
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, (name, hist) in enumerate(reg_histories.items()):
    ax = axes[i]
    ep = range(1, len(hist['train_acc']) + 1)
    ax.plot(ep, hist['train_acc'], label='Train', color='#2196F3')
    ax.plot(ep, hist['val_acc'], label='Val', color='#F44336')
    gap = hist['train_acc'][-1] - hist['val_acc'][-1]
    ax.fill_between(ep, hist['val_acc'], hist['train_acc'], alpha=0.2, color='orange', label=f'Gap={gap:.3f}')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Overfitting on 500 Samples: Dropout Reduces Generalisation Gap', fontsize=12)
plt.tight_layout()
plt.show()

## 3.5 — Building a Complete CNN: Architecture Summary

We now have all the building blocks. A complete classification CNN is:

Input → [Conv → BN → ReLU → MaxPool] × N → GAP → Dropout → FC → Softmax

Design checklist:
  - ✓ Use 3×3 kernels (efficient, good receptive field growth)
  - ✓ Use 'same' padding (kernel_size//2) to control spatial dims explicitly
  - ✓ Double channels with each MaxPool (classic VGG/AlexNet pattern)
  -  ✓ BatchNorm after every Conv (before ReLU or after — both work)
  - ✓ Global Average Pool instead of Flatten+FC (fewer params, better generalisation)
  - ✓ Dropout(0.3-0.5) before the final linear layer
  - ✓ bias=False in Conv when followed by BatchNorm

In [ ]:
# The complete SimpleCNN for EuroSAT — well-commented reference implementation
class EuroSATCNN(nn.Module):
    """
    Simple 3-block CNN for EuroSAT classification.
    Architecture: Input(3,64,64) → 3×ConvBlock → GAP → FC(10)
    
    Key choices explained:
    - bias=False in Conv (redundant with BatchNorm)
    - BN before ReLU (original paper order)
    - GAP instead of Flatten+Dense (fewer params, less overfitting)
    - Dropout(0.3) as final regulariser
    """
    def __init__(self, in_channels=3, num_classes=10, base=32, dropout=0.3):
        super().__init__()
        
        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),   # halve spatial dims
            )
        
        self.features = nn.Sequential(
            conv_block(in_channels, base),       # 64→32, channels: 3→32
            conv_block(base, base * 2),          # 32→16, channels: 32→64
            conv_block(base * 2, base * 4),      # 16→8,  channels: 64→128
        )
        
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),             # (B, 128, 8, 8) → (B, 128, 1, 1)
            nn.Flatten(),                        # → (B, 128)
            nn.Dropout(dropout),
            nn.Linear(base * 4, num_classes),    # → (B, 10)
        )
    
    def forward(self, x):
        return self.head(self.features(x))

model = EuroSATCNN(in_channels=3, num_classes=10)
x = torch.randn(4, 3, 64, 64)
out = model(x)
print(f'Input:  {x.shape}')
print(f'Output: {out.shape}  (4 samples, 10 class logits each)')
print(f'\nModel parameters: {sum(p.numel() for p in model.parameters()):,}')
print()

# Trace shapes through the network
x_trace = torch.randn(1, 3, 64, 64)
print('Shape trace through features:')
for i, layer in enumerate(model.features):
    x_trace = layer(x_trace)
    print(f'  After block {i+1}: {x_trace.shape}')

## Practical Exercises

### Exercise 3.1 — Architecture Ablation
Train 3 variants of EuroSATCNN for 5 epochs each:
  A) base=16 (narrow)
  B) base=32 (default)
  C) base=64 (wide)
Compare: parameters, training time per epoch, final validation accuracy.

### Exercise 3.2 — Activation Comparison
Replace nn.ReLU with nn.LeakyReLU(0.1) and nn.GELU in EuroSATCNN.
Train for 5 epochs each and compare learning curves.

### Exercise 3.3 — Batch Size Effect
Train the same model with batch_size in [16, 32, 64, 128].
Plot training loss at each batch size. Why does large batch size hurt?

### Mini-Project 3
Build a CNN that achieves >70% validation accuracy on EuroSAT
using ONLY the building blocks from this chapter (no pretrained models).
Submit: architecture diagram, learning curves, and confusion matrix.

In [ ]:
print('Chapter 3 Summary:')
print('  MaxPool2d: spatial downsampling, translation invariance')
print('  ReLU: simple, effective, default activation (watch dying ReLU)')
print('  BatchNorm: normalises activations, faster training, implicit regularisation')
print('  Dropout: prevents co-adaptation, use in FC layers')
print('  Global Average Pool: replaces Flatten+Dense, fewer params')
print()
print('In Chapter 4: we handle REAL Sentinel-2 geospatial data —')
print('loading GeoTIFFs, handling CRS, tiling, and building spectral indices.')